# 📗 SQL 조회와 결합 — 원하는 것만 꺼내 리포트로

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

지난 노트북에서 설계도를 그리고 표를 만들어 데이터를 넣었습니다. 이제 그 안에서 **필요한 것만 꺼내는 법**을 배웁니다. 조건으로 거르고, 줄 세우고, 묶어 계산하고, 흩어진 표를 이어서 리포트 한 장을 만드는 데까지 갑니다.

**오늘 이 노트북을 마치면**

- `SELECT` · `WHERE` 로 원하는 열과 행만 꺼낼 수 있습니다.
- `ORDER BY` · `LIMIT` 으로 줄 세워 필요한 만큼만 가져올 수 있습니다.
- `GROUP BY` · `HAVING` 으로 그룹마다 계산하고 조건을 걸 수 있습니다.
- `JOIN` · `LEFT JOIN` 으로 여러 표를 이어 한 줄로 읽어 낼 수 있습니다.
- **서브쿼리**로 질의 안에 질의를 넣고, 바깥 행을 가리키는 **상관 서브쿼리**까지 쓸 수 있습니다.
- 자주 거르는 열에 **인덱스**를 만들어 두는 이유를 설명할 수 있습니다.

## ⏪ 복습 — 지난 노트북에서

- 표는 **열**(정보의 종류)과 **행**(데이터 한 건)으로 이뤄집니다.
- **타입**과 **제약조건**(`NOT NULL` · `UNIQUE` · `CHECK` · `DEFAULT`)이 값의 종류와 범위를 정합니다.
- **기본키**로 행을 콕 집고, **외래키**로 표를 잇습니다. 외래키는 N 쪽에 둡니다.
- `CREATE TABLE` 로 표를 만들고 `INSERT` 로 행을 넣었습니다.
- `UPDATE` · `DELETE` 로 고치고 지웠습니다 — **같은 `WHERE` 로 `SELECT` 를 먼저**, 위험한 문장은 `BEGIN` … `ROLLBACK` 으로 묶어서.

이번엔 데이터를 **꺼내는** 쪽입니다. 매번 표를 새로 만들지 않도록, 아래 리셋 셀이 지난 시간에 만든 것과 같은 구조의 표를 데이터까지 채워 줍니다.

| 표 | 무엇 | 행 수 |
|---|---|---|
| `customers` | 고객 | 5 |
| `products` | 상품(정가) | 6 |
| `orders` | 주문 | 8 |
| `reviews` | 후기 | 8 |

In [ ]:
# [제공 코드] — sqlite 실습 준비 (내용은 이해하지 않아도 됩니다 — 실행만 하세요)
import sqlite3
from pathlib import Path

import pandas as pd

ROOT = Path(".") if Path("data").is_dir() else Path("..")
DB_PATH = ROOT / "output" / "shop.db"
DB_PATH.parent.mkdir(exist_ok=True)

_conn = None


def get_conn():
    """실습용 sqlite 연결을 하나만 만들어 계속 재사용합니다."""
    global _conn
    if _conn is None:
        _conn = sqlite3.connect(DB_PATH, isolation_level=None)  # 실행하는 즉시 저장(자동 커밋)
        _conn.execute("pragma foreign_keys = on")               # 외래키 검사를 켭니다
    return _conn


def reset_db(script=None):
    """실습 DB를 처음 상태로 되돌립니다. 언제 몇 번을 다시 실행해도 안전합니다."""
    global _conn
    if _conn is not None:
        _conn.close()
        _conn = None
    DB_PATH.unlink(missing_ok=True)
    conn = get_conn()
    if script is not None:
        conn.executescript((ROOT / "data" / script).read_text(encoding="utf-8"))
        conn.execute("pragma foreign_keys = on")
    return conn


def run_sql(sql):
    """결과가 없는 SQL(CREATE·INSERT·UPDATE·DELETE 등)을 실행합니다."""
    get_conn().execute(sql)


def run_query(sql):
    """SELECT 결과를 pandas DataFrame 으로 돌려줍니다."""
    cur = get_conn().execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])


print("sqlite 준비 완료 —", DB_PATH)

In [ ]:
# [제공 코드] — 실습 테이블 전체 리셋 (언제든 다시 실행하면 처음 상태로 돌아갑니다)
reset_db("setup_shop.sql")

for t in ["customers", "products", "orders", "reviews"]:
    n = run_query(f"SELECT count(*) AS n FROM {t}")["n"][0]
    print(f"{t}: {n}행")

## 1. SELECT 와 WHERE — 필요한 열, 필요한 행만

표 전체를 꺼내는 일은 실무에서 거의 없습니다. 6만 건짜리 표를 통째로 받아 봐야 읽을 수 없으니까요. **어떤 열을** 볼지 고르고, **어떤 행만** 남길지 거릅니다.

```
SELECT 열들 FROM 표 WHERE 조건;
```

- `*` 는 모든 열, 아니면 필요한 열만 콤마로 나열합니다.
- 열 자리에는 **계산식**도 올 수 있고, `AS` 로 **별칭**을 붙여 결과 이름을 바꿉니다.
- `WHERE` 뒤 비교연산자: `=` · `<>`(같지 않다) · `>` · `<` · `>=` · `<=`
- 문자열은 작은따옴표로 감쌉니다. 숫자는 그대로 씁니다.

In [ ]:
# 필요한 열만, 조건에 맞는 행만
display(run_query("SELECT name, city FROM customers WHERE city = '서울'"))

# 계산 열과 별칭 — 금액에 부가세 10% 를 더한 값을 새 이름으로
display(run_query("""
SELECT product,
       amount,
       amount * 1.1 AS amount_with_tax
FROM orders
WHERE amount >= 100000
"""))

### 🖐️ 함께 따라하기 — 상품표에서 조건과 별칭 함께 쓰기

`products` 표에서 **정가가 10만원 이상인 상품**만 골라, 상품명(`name`)과 **정가의 10% 를 계산한 값**을 `discount` 라는 별칭으로 함께 조회하세요.

**확인 기준**: 3행(노트북·모니터·헤드셋)이 나옵니다.

In [ ]:
# products 에서 정가 10만원 이상인 상품의 name 과 정가의 10% 를 discount 별칭으로 조회하세요.

### ✅ 바로 확인 퀴즈

`SELECT name FROM customers` 와 `SELECT * FROM customers` 의 차이는 무엇일까요?

- **A.** 앞은 행이 적게, 뒤는 행이 많이 나옵니다
- **B.** 앞은 열 하나만, 뒤는 모든 열이 나옵니다
- **C.** 앞은 정렬되고 뒤는 정렬되지 않습니다
- **D.** 둘은 똑같습니다

<details><summary>정답 보기</summary>

**B** — `SELECT` 뒤는 **어떤 열을 볼지**를 정합니다. 행을 줄이는 것은 `WHERE` 의 일입니다.

</details>

## 2. ORDER BY · LIMIT · OFFSET — 줄 세우고 잘라 가져오기

조회 결과의 **순서는 아무것도 보장되지 않습니다.** 넣은 순서대로 나오는 것처럼 보여도 우연입니다. 순서가 필요하면 반드시 `ORDER BY` 로 말해 줘야 합니다.

| 문법 | 하는 일 |
|---|---|
| `ORDER BY 열` | 그 열 기준 오름차순(작은 것부터) |
| `ORDER BY 열 DESC` | 내림차순(큰 것부터) |
| `ORDER BY A, B DESC` | A 로 먼저 세우고, 같으면 B 내림차순 |
| `LIMIT n` | 위에서 n 개만 |
| `OFFSET m` | 앞의 m 개를 건너뛰고 |

행이 줄어드는 순서는 **거르기(`WHERE`) → 정렬(`ORDER BY`) → 자르기(`LIMIT`)** 입니다.

In [ ]:
# 비싼 주문 3건
display(run_query("SELECT product, amount FROM orders ORDER BY amount DESC LIMIT 3"))

# 다중 정렬 — 고객 번호 오름차순, 같은 고객 안에서는 금액 큰 순
display(run_query("SELECT customer_id, product, amount FROM orders "
                  "ORDER BY customer_id, amount DESC"))

# 4~6등만 보기 — 앞의 3개를 건너뛰고 3개
display(run_query("SELECT product, amount FROM orders ORDER BY amount DESC LIMIT 3 OFFSET 3"))

### 🖐️ 함께 따라하기 — 후기에서 낮은 평점부터

`reviews` 표에서 **평점이 낮은 순**으로 세우고, 같은 평점이면 **후기 번호(`id`) 오름차순**으로 정렬해 `id` · `product` · `rating` 을 조회하세요. 그중 **위에서 3건만** 가져옵니다.

**확인 기준**: 첫 행이 평점 2점짜리 키보드 후기입니다.

In [ ]:
# reviews 를 평점 낮은 순(같으면 id 오름차순)으로 세워 위에서 3건만 조회하세요.

### ✅ 바로 확인 퀴즈

`ORDER BY` 없이 `SELECT * FROM orders LIMIT 3` 을 실행하면 어떤 3건이 나올까요?

- **A.** 금액이 가장 큰 주문 3건
- **B.** 가장 먼저 넣은 주문 3건
- **C.** 무엇이 나올지 보장되지 않습니다
- **D.** 실행할 때마다 무작위로 달라집니다

<details><summary>정답 보기</summary>

**C** — 정렬을 말해 주지 않으면 **어떤 순서로 나올지 보장되지 않습니다.** 지금 넣은 순서대로 보이더라도 데이터가 쌓이면 달라질 수 있습니다. 순서가 중요하면 반드시 `ORDER BY` 를 씁니다.

</details>

## 3. 조건 심화 — 여러 조건 · 범위 · 목록 · 패턴 · 빈 값

실제 질문은 조건이 겹쳐 있습니다. "3만원에서 10만원 사이면서 상품 이름에 '보드'가 들어간 주문" 처럼요.

지난 노트북에서 `LIKE` 의 `%`·`_` 와 `NOT` 을 잠깐 맛봤습니다. 여기서는 그것들을 **범위·목록·빈 값 조건과 함께** 제대로 씁니다.

| 문법 | 하는 일 | 예 |
|---|---|---|
| `AND` · `OR` · `NOT` | 조건 묶기 | `city = '서울' AND amount >= 50000` |
| `BETWEEN a AND b` | a 부터 b 까지, **양끝 포함** | `amount BETWEEN 30000 AND 100000` |
| `IN (…)` · `NOT IN (…)` | 목록 중 하나라도 맞으면 · 목록에 하나도 없으면 | `city IN ('서울', '부산')` |
| `LIKE '패턴'` | 글자 패턴 — `%` 는 아무 글자 여러 개, `_` 는 한 글자 | `product LIKE '%보드'` |
| `IS NULL` · `IS NOT NULL` | 빈 값인가 | `city IS NULL` |
| `DISTINCT` | 중복 없이 종류만 | `SELECT DISTINCT city FROM customers` |

> **괄호를 꼭 쓰세요.** `AND` 가 `OR` 보다 먼저 계산됩니다. `A OR B AND C` 는 `A OR (B AND C)` 로 읽힙니다 — 의도와 다르기 쉽습니다.

> **NULL 은 0 도 빈 글자도 아닙니다.** 값이 없다는 표시라서 `= NULL` 로는 절대 찾을 수 없고 `IS NULL` 로만 찾습니다.

In [ ]:
# 1) 여러 조건 — 괄호로 우선순위를 분명히
display(run_query("""
SELECT name, city
FROM customers
WHERE (city = '서울' OR city = '부산') AND name <> '김민수'
"""))

# 2) 범위와 목록
display(run_query("SELECT product, amount FROM orders "
                  "WHERE amount BETWEEN 30000 AND 100000"))
display(run_query("SELECT name, city FROM customers WHERE city IN ('서울', '대구')"))

In [ ]:
# 3) 패턴과 빈 값
display(run_query("SELECT DISTINCT product FROM orders WHERE product LIKE '%보드'"))
display(run_query("SELECT id, name, city FROM customers WHERE city IS NULL"))

# 4) 중복 없이 종류만
display(run_query("SELECT DISTINCT city FROM customers"))
print("NULL 도 하나의 '종류'로 함께 나옵니다.")

### 🖐️ 함께 따라하기 — 후기에서 조건 세 가지 겹치기

`reviews` 표에서 아래 조건을 **모두** 만족하는 후기를 찾아 `product` · `rating` · `comment` 를 조회하세요.

- 평점이 3점 이상 5점 이하
- 후기 내용(`comment`)이 비어 있지 않음
- 상품 이름이 '마우스' 또는 '키보드'

**확인 기준**: 3행이 나옵니다.

In [ ]:
# reviews 에서 위 세 조건을 모두 만족하는 후기를 조회하세요.

### ✅ 바로 확인 퀴즈

도시를 적지 않은 고객을 찾으려고 `WHERE city = NULL` 이라고 썼습니다. 결과는?

- **A.** 도시가 빈 고객이 나옵니다
- **B.** 모든 고객이 나옵니다
- **C.** 한 행도 나오지 않습니다
- **D.** 문법 오류가 납니다

<details><summary>정답 보기</summary>

**C** — NULL 은 '값이 없다'는 표시라서 무엇과도 **같다고 판정되지 않습니다.** 빈 값은 오직 `IS NULL` 로만 찾습니다.

</details>

## 4. 집계 — 여러 행을 한 값으로

주문이 6건이면 눈으로 세도 됩니다. 6만 건이면요? **집계 함수**는 표를 훑어 값 하나로 답합니다.

| 함수 | 무엇을 |
|---|---|
| `count(*)` | 행이 몇 개인지 (빈 칸이 있어도 셉니다) |
| `count(열)` | 그 열이 **비어 있지 않은** 행만 셉니다 |
| `count(DISTINCT 열)` | 그 열의 **서로 다른 값**이 몇 가지인지 |
| `sum(열)` · `avg(열)` | 합계 · 평균 |
| `max(열)` · `min(열)` | 가장 큰 값 · 가장 작은 값 |

`count(*)` 와 `count(열)` 의 차이가 실무에서 가장 자주 헷갈립니다 — 직접 확인해 봅시다.

In [ ]:
# count(*) 와 count(열) 의 차이 — city 가 비어 있는 고객이 있습니다.
display(run_query("""
SELECT count(*)              AS 전체고객,
       count(city)           AS 도시적은고객,
       count(DISTINCT city)  AS 도시종류
FROM customers
"""))

# 매출 요약 한 줄
display(run_query("""
SELECT count(*)   AS 주문건수,
       sum(amount) AS 총매출,
       avg(amount) AS 평균금액,
       max(amount) AS 최고금액,
       min(amount) AS 최저금액
FROM orders
"""))

### 🖐️ 함께 따라하기 — 후기 평점 요약

`reviews` 표에서 **후기 수 · 내용이 있는 후기 수 · 평균 평점 · 최고 평점 · 최저 평점**을 한 번에 구하세요. 별칭은 각각 `후기수` · `내용있는후기` · `평균평점` · `최고평점` · `최저평점` 으로 붙입니다.

**확인 기준**: 후기수 8 · 내용있는후기 7 이 나옵니다.

In [ ]:
# reviews 의 후기 수·내용 있는 후기 수·평균·최고·최저 평점을 한 번에 구하세요.

### ✅ 바로 확인 퀴즈

고객 5명 중 1명은 도시를 적지 않았습니다. `SELECT count(city) FROM customers` 의 결과는?

- **A.** 5
- **B.** 4
- **C.** 1
- **D.** 0

<details><summary>정답 보기</summary>

**B** — `count(열)` 은 그 열이 **비어 있지 않은 행만** 셉니다. 행 전체를 세려면 `count(*)` 를 씁니다.

</details>

## 5. GROUP BY · HAVING — 그룹마다 계산하고 그룹에 조건 걸기

"전체 매출"이 아니라 "**고객별** 매출"이 필요할 때 씁니다. `GROUP BY 열` 은 같은 값을 가진 행끼리 묶어, 묶음마다 집계 함수를 계산합니다.

```
SELECT 기준열, 집계함수(...)
FROM 표
WHERE  행 조건        -- 묶기 전에 행을 거른다
GROUP BY 기준열
HAVING 그룹 조건      -- 묶은 뒤에 그룹을 거른다
ORDER BY ...
```

**`WHERE` 와 `HAVING` 의 차이**가 핵심입니다. `WHERE` 는 **묶기 전** 행을 거르고, `HAVING` 은 **묶은 뒤** 그룹을 거릅니다. 그래서 `HAVING` 에는 집계 함수가 올 수 있습니다.

**적는 순서와 실행 순서는 다릅니다.**

| 적는 순서 | 실행 순서 |
|---|---|
| `SELECT` → `FROM` → `WHERE` → `GROUP BY` → `HAVING` → `ORDER BY` | `FROM` → `WHERE` → `GROUP BY` → `HAVING` → **`SELECT`** → `ORDER BY` |

![SQL 실행 순서](images/sql_실행순서.png)

`SELECT` 는 맨 앞에 적지만 거의 마지막에 실행됩니다. 그래서 **`WHERE` 에는 아직 만들어지지 않은 것을 쓸 수 없습니다** — 묶기 전이라 집계 함수(`count(*)`)를 쓸 수 없고, 묶은 뒤에 거르려면 `HAVING` 을 씁니다. 바로 아래 셀에서 **실행해서** 확인합니다.

> **별칭도 같은 이유로 `WHERE` 에서는 못 씁니다** — 별칭은 `SELECT` 를 처리할 때 비로소 생기니까요. 다만 sqlite 는 별칭에 한해 유독 관대해서 그냥 통과시켜 줍니다. 표준 SQL 을 따르는 데이터베이스는 거절하므로, 별칭은 `ORDER BY` 에서 쓰는 습관을 들이세요. 이것도 아래 셀에서 직접 돌려 봅니다.

In [ ]:
# 실행 순서를 실행으로 확인합니다.

# 1) WHERE 에 집계 함수 — 아직 묶기 전이라 셀 것이 없습니다. 오류가 납니다.
try:
    run_query("""
    SELECT customer_id, count(*) AS 주문건수
    FROM orders
    WHERE count(*) >= 2
    GROUP BY customer_id
    """)
except sqlite3.OperationalError as e:
    print("1) WHERE 에 집계 함수 -> 오류:", e)

# 2) 같은 조건을 HAVING 에 두면 됩니다 — 묶은 뒤라 셀 것이 있습니다.
print("\n2) HAVING 으로 옮기면:")
display(run_query("""
SELECT customer_id, count(*) AS 주문건수
FROM orders
GROUP BY customer_id
HAVING count(*) >= 2
"""))

In [ ]:
# 3) SELECT 에서 붙인 별칭은 ORDER BY 에서는 쓸 수 있습니다 — ORDER BY 가 SELECT 뒤에 실행되니까요.
display(run_query("""
SELECT product, amount * 1.1 AS 세금포함
FROM orders
ORDER BY 세금포함 DESC
LIMIT 3
"""))

# 4) 같은 별칭을 WHERE 에 쓰면? 표준 SQL·PostgreSQL 은 오류입니다.
try:
    df = run_query("""
    SELECT product, amount * 1.1 AS 세금포함
    FROM orders
    WHERE 세금포함 >= 100000
    """)
    print("sqlite 는 통과했습니다 —", len(df), "행")
except sqlite3.OperationalError as e:
    print("오류:", e)

print("\n참고 — 위 4번은 sqlite 라서 통과한 것입니다. 같은 질의를 PostgreSQL 에 보내면")
print('   이런 오류가 돌아옵니다:  column "세금포함" does not exist')
print("   그러니 별칭은 WHERE 가 아니라 ORDER BY 에서 쓰는 습관을 들이세요.")

In [ ]:
# 고객별 주문 건수와 매출
display(run_query("""
SELECT customer_id,
       count(*)    AS 주문건수,
       sum(amount) AS 매출
FROM orders
GROUP BY customer_id
ORDER BY 매출 DESC
"""))

In [ ]:
# WHERE 와 HAVING 을 함께 — 3만원 이상 주문만 세고(WHERE), 그중 2건 이상인 고객만(HAVING)
display(run_query("""
SELECT customer_id,
       count(*)    AS 주문건수,
       sum(amount) AS 매출
FROM orders
WHERE amount >= 30000
GROUP BY customer_id
HAVING count(*) >= 2
ORDER BY 매출 DESC
"""))

### 🖐️ 함께 따라하기 — 상품별 후기 통계

`reviews` 표를 **상품별로 묶어** 후기 수와 평균 평점을 구하되, **후기가 2건 이상인 상품만** 남기세요. 별칭은 `후기수` · `평균평점` 이고, 평균 평점이 높은 순으로 정렬합니다.

**확인 기준**: 3행(노트북·마우스·키보드)이 나옵니다.

In [ ]:
# reviews 를 상품별로 묶어 후기 수·평균 평점을 구하고, 2건 이상인 상품만 평균 평점 높은 순으로.

### ✅ 바로 확인 퀴즈

"주문 금액이 10만원 이상인 건만 골라, 고객별 건수를 세고 싶다"면 조건을 어디에 쓸까요?

- **A.** `HAVING amount >= 100000`
- **B.** `WHERE amount >= 100000`
- **C.** `SELECT` 뒤에
- **D.** `ORDER BY` 뒤에

<details><summary>정답 보기</summary>

**B** — **행 하나하나**에 걸리는 조건이므로 묶기 전인 `WHERE` 입니다. `HAVING` 은 `count(*) >= 2` 처럼 **묶은 결과**에 거는 조건입니다.

</details>

## 6. JOIN — 번호만 적힌 표에 이름 붙이기

`orders` 를 아무리 들여다봐도 **누가** 샀는지는 안 나옵니다. 거기엔 `customer_id` 번호만 있으니까요. 이름은 `customers` 에 있습니다. 두 표를 이어야 한 줄로 읽힙니다.

```
SELECT 열들
FROM 표A
JOIN 표B ON 표A.기본키 = 표B.외래키;
```

`ON` 에는 **외래키 = 기본키** 조건을 적습니다. 같은 값을 가진 행끼리 옆으로 붙어 한 행이 됩니다. 표 이름이 길면 `customers c` 처럼 **별칭**을 붙여 `c.name` 으로 짧게 씁니다.

![조인 두 가지](images/조인_두_가지.png)

In [ ]:
# 고객 이름이 붙은 주문 목록
display(run_query("""
SELECT c.name, o.product, o.amount
FROM customers c
JOIN orders o ON o.customer_id = c.id
ORDER BY o.amount DESC
"""))

print("주문 8건이 각자 주인을 찾아 8행이 되었습니다.")

### 🖐️ 함께 따라하기 — 후기에 고객 이름 붙이기

`reviews` 와 `customers` 를 이어, **고객 이름 · 상품 · 평점 · 후기 내용**을 조회하세요. 평점이 높은 순, 같으면 고객 이름 오름차순으로 정렬합니다.

**확인 기준**: 8행이 나오고 첫 행의 평점이 5점입니다.

In [ ]:
# reviews 와 customers 를 이어 고객 이름·상품·평점·내용을 평점 높은 순으로 조회하세요.

### ✅ 바로 확인 퀴즈

고객 5명(그중 2명은 주문 0건)과 주문 8건을 `JOIN` 으로 이으면 결과는 몇 행일까요?

- **A.** 5행
- **B.** 8행
- **C.** 10행
- **D.** 13행

<details><summary>정답 보기</summary>

**B** — `JOIN`(= `INNER JOIN`)은 **양쪽에 짝이 있는 행만** 남깁니다. 주문 8건이 각자 주인을 찾아 8행이 되고, 주문이 0건인 고객 2명은 결과에서 사라집니다.

</details>

## 7. LEFT JOIN — 짝이 없는 행까지 남기기

방금 결과에서 **주문이 0건인 고객은 아예 사라졌습니다.** "한 명도 빠지지 않는 고객별 매출표"를 만들어야 한다면 곤란하죠.

`LEFT JOIN` 은 `FROM` 뒤에 적은 **왼쪽 표의 행을 하나도 버리지 않습니다.** 짝이 없으면 오른쪽 자리를 `NULL` 로 채웁니다.

그 `NULL` 을 그대로 두면 합계가 `NULL` 이 되니, **`coalesce(값, 대체값)`** 으로 0 을 채웁니다.

In [ ]:
# 주문이 0건인 고객까지 남습니다 — NULL 로 채워진 자리를 확인하세요.
display(run_query("""
SELECT c.name, o.product, o.amount
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.id
ORDER BY c.id
"""))

In [ ]:
# 한 명도 빠지지 않는 고객별 매출 — NULL 을 0 으로 바꿔서
display(run_query("""
SELECT c.name,
       count(o.id)                  AS 주문건수,
       coalesce(sum(o.amount), 0)   AS 매출
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.id
GROUP BY c.id, c.name
ORDER BY 매출 DESC
"""))
print("count(o.id) 는 짝이 없는 행을 세지 않아 0 이 됩니다 — count(*) 였다면 1 이 됩니다.")

**`LEFT JOIN` 에서는 조건을 어디에 적는지가 결과를 바꿉니다.** "4월 주문만 세되 **고객은 한 명도 빠뜨리지 말라**"는 요구를 두 방식으로 써 보고 나란히 견줘 봅시다. 같은 기간 조건인데 `WHERE` 에 두면 **2행**, `ON` 에 두면 **5행**이 나옵니다 — 고객 5명 중 4월에 주문한 사람은 두 명뿐이기 때문입니다.

In [ ]:
# 가) 기간 조건을 WHERE 에 둔 경우
where_version = run_query("""
SELECT c.name, count(o.id) AS 주문건수
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.id
WHERE o.ordered_at BETWEEN '2026-04-01' AND '2026-04-30'
GROUP BY c.id, c.name
ORDER BY c.id
""")
display(where_version)
print("남은 고객 수:", len(where_version))

In [ ]:
# 나) 같은 조건을 LEFT JOIN 의 ON 절로 옮긴 경우
on_version = run_query("""
SELECT c.name, count(o.id) AS 주문건수
FROM customers c
LEFT JOIN orders o
       ON o.customer_id = c.id
      AND o.ordered_at BETWEEN '2026-04-01' AND '2026-04-30'
GROUP BY c.id, c.name
ORDER BY c.id
""")
display(on_version)
print("남은 고객 수:", len(on_version))
print("-> WHERE 는 조인이 다 끝난 뒤에 남은 행을 거릅니다.")
print("   4월이 아닌 주문 행은 날짜 조건에 걸려 사라지고, 주문이 아예 없는 고객의 NULL 행도")
print("   조건과 맞지 않아 사라집니다. 그래서 4월에 주문한 고객만 남습니다.")
print("   ON 에 적으면 짝을 맺을 때만 쓰이므로, 짝이 없는 고객은 NULL 을 달고 그대로 남습니다.")

> 아래 셀은 `orders` 와 `products` 를 **상품 이름**으로 잇습니다. 실무라면 `orders` 에 `product_id` 를 두고 그 번호로 잇는 것이 맞습니다(지난 노트북 5절). 여기서는 조인 조건이 꼭 기본키·외래키일 필요는 없다는 것을 보이려고 이름으로 이었습니다.

In [ ]:
# 표 3개 잇기 — 주문에 고객 이름과 상품 정가까지 붙입니다.
display(run_query("""
SELECT c.name, o.product, o.amount, p.list_price,
       p.list_price - o.amount AS 할인액
FROM orders o
JOIN customers c ON c.id = o.customer_id
JOIN products  p ON p.name = o.product
ORDER BY 할인액 DESC
"""))

### 🖐️ 함께 따라하기 — 후기가 하나도 없는 상품까지 보이게

`products` 를 왼쪽에 두고 `reviews` 를 이어, **모든 상품**의 상품명 · 후기 수 · 평균 평점을 구하세요. 후기가 없는 상품은 후기 수가 `0` 이어야 합니다.

- 별칭은 `후기수` · `평균평점`
- `평균평점` 이 `NULL` 인 상품(후기 없음)도 그대로 두세요
- 후기 수가 많은 순으로 정렬

**확인 기준**: 6행이 나오고, 아직 후기가 없는 헤드셋의 후기수가 0 입니다.

In [ ]:
# products 를 왼쪽에 두고 reviews 를 LEFT JOIN 해 상품별 후기 수·평균 평점을 구하세요.

### ✅ 바로 확인 퀴즈

`LEFT JOIN` 결과에서 짝이 없는 자리의 값은 무엇일까요?

- **A.** 0
- **B.** 빈 문자열
- **C.** NULL
- **D.** 그 행 자체가 사라집니다

<details><summary>정답 보기</summary>

**C** — 짝이 없으면 오른쪽 표의 열들은 `NULL` 로 채워집니다. 합계를 낼 때는 `coalesce(sum(...), 0)` 으로 0 을 채워 줍니다.

</details>

### 집계 결과를 표로 남기기 — `INSERT INTO … SELECT`

`INSERT` 의 `VALUES` 자리에는 값 대신 **`SELECT` 를 통째로** 놓을 수 있습니다. 그러면 조회해서 나온 행이 그대로 다른 표에 담깁니다. 특히 `SELECT` 자리에 **방금 만든 집계 질의**를 넣으면 "이번 달 우수고객 명단"처럼 계산 결과를 표로 남길 수 있습니다.

```sql
INSERT INTO 받을표 (열1, 열2) SELECT 열A, 열B FROM 준표 WHERE 조건;
```

**열 짝은 이름이 아니라 적은 순서로 지어집니다** — 넣을 열 세 개와 `SELECT` 가 돌려주는 열 세 개가 순서까지 1:1 로 맞아야 합니다. 순서를 바꿔 적어도 타입만 맞으면 오류 없이 엉뚱한 칸에 들어가니, 넣기 전에 `SELECT` 만 따로 돌려 눈으로 맞춰 보세요.

In [ ]:
# 매출 10만원 이상인 고객만 골라 별도 표로 남깁니다.
run_sql("""
CREATE TABLE sales_report (
    customer_id int  PRIMARY KEY,
    name        text NOT NULL,
    total       int  NOT NULL
) STRICT
""")

run_sql("""
INSERT INTO sales_report (customer_id, name, total)
SELECT c.id, c.name, coalesce(sum(o.amount), 0)
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.id
GROUP BY c.id, c.name
HAVING coalesce(sum(o.amount), 0) >= 100000
""")

display(run_query("SELECT * FROM sales_report ORDER BY total DESC"))

run_sql("DROP TABLE sales_report")   # 확인이 끝났으니 연습용 표는 치웁니다

### 인덱스 — 자주 거르는 열에 미리 찾아보기표를 만들어 둔다

조인 이야기는 여기까지입니다. 다음 절로 넘어가기 전에, 지금까지 쓴 `WHERE` · `JOIN` 이 **데이터가 많아지면** 어떻게 되는지 한 가지만 짚고 갑시다.

지금은 주문이 8건이라 어떤 조건이든 눈 깜짝할 사이에 끝납니다. 하지만 800만 건이라면 `WHERE customer_id = 3` 하나에도 데이터베이스가 전체를 처음부터 끝까지 훑어야 합니다.

**인덱스(index)** 는 책 뒤의 찾아보기와 같습니다 — 그 열의 값을 미리 정렬해 두고 "이 값은 몇 번째 행"을 적어 둡니다. 그러면 전체를 훑지 않고 바로 그 자리로 갑니다. 대신 행을 넣고 고칠 때마다 찾아보기도 함께 갱신해야 해서, **자주 거르거나 잇는 열에만** 만듭니다.

In [ ]:
# 자주 조건으로 쓰는 열에 인덱스를 만들어 둡니다.
run_sql("CREATE INDEX orders_customer_id_idx ON orders(customer_id)")
display(run_query("SELECT name, tbl_name FROM sqlite_master WHERE type = 'index' AND sql IS NOT NULL"))
print("-> 질의는 그대로 쓰면 됩니다. 데이터베이스가 알아서 이 찾아보기를 씁니다.")

다음 노트북에서 만나는 **HNSW** 도 인덱스입니다 — 숫자나 글자가 아니라 **벡터**를 위한 찾아보기라는 점만 다릅니다.

## 8. 서브쿼리 — 질의 안의 질의

"**평균보다 비싼** 주문"을 찾으려면 평균을 먼저 알아야 합니다. 그런데 평균도 질의로 구하죠. 이렇게 **질의 결과를 다른 질의 안에 넣는 것**이 서브쿼리입니다. 괄호로 감싸 씁니다.

| 모양 | 언제 |
|---|---|
| `WHERE 열 > (SELECT avg(...) FROM …)` | 결과가 **값 하나**일 때 |
| `WHERE 열 IN (SELECT … FROM …)` | 결과가 **목록**일 때 |
| `WHERE 열 NOT IN (SELECT … FROM …)` | 그 **목록에 없는** 것만 남길 때 |
| `(SELECT … WHERE 안쪽.열 = 바깥.열)` | 바깥 행마다 값을 따로 구할 때 (**상관 서브쿼리**) |

> 서브쿼리는 안쪽부터 읽습니다. 안쪽 질의를 따로 실행해 값을 확인한 뒤 바깥에 끼워 넣으면 훨씬 덜 헷갈립니다.

In [ ]:
# 1) 안쪽 질의를 먼저 확인합니다.
display(run_query("SELECT avg(amount) AS 평균금액 FROM orders"))

# 2) 그 값을 바깥 조건에 끼워 넣습니다.
display(run_query("""
SELECT product, amount
FROM orders
WHERE amount > (SELECT avg(amount) FROM orders)
ORDER BY amount DESC
"""))

In [ ]:
# 목록을 돌려주는 서브쿼리 — 주문한 적이 있는 고객만
display(run_query("""
SELECT name, city
FROM customers
WHERE id IN (SELECT DISTINCT customer_id FROM orders)
ORDER BY id
"""))

In [ ]:
# NOT IN — 목록에 '없는' 것만 남깁니다. 주문한 적이 한 번도 없는 고객을 찾습니다.
display(run_query("""
SELECT name, city
FROM customers
WHERE id NOT IN (SELECT DISTINCT customer_id FROM orders)
ORDER BY id
"""))

> **`NOT IN` 의 목록에 `NULL` 이 섞이면 결과가 통째로 빕니다.** `NOT IN` 은 "모든 값과 다르다"를 확인하는데, `NULL` 과는 같은지 다른지 판정 자체가 안 되기 때문입니다(3절의 `= NULL` 과 같은 이유). 그래서 서브쿼리로 목록을 만들 때는 그 열이 `NOT NULL` 인지 확인하거나, `WHERE 열 IS NOT NULL` 을 안쪽에 붙여 둡니다. 위 `orders.customer_id` 는 `NOT NULL` 이라 안전합니다.

### 상관 서브쿼리 — 바깥 행을 가리키는 서브쿼리

위의 두 서브쿼리는 **혼자서도 실행됩니다.** 따로 떼어 내 돌려 보면 값 하나 또는 목록 하나가 나오고, 바깥 질의는 그 결과를 한 번만 받아 씁니다.

그런데 서브쿼리 안에서 **바깥 질의의 행을 가리킬** 수도 있습니다(`o.customer_id = c.id` 처럼). 이런 서브쿼리는 혼자서는 실행되지 않고, **바깥 행 하나마다 다시 계산됩니다.** 고객마다 다른 값을 붙여야 할 때 쓰는 모양입니다.

In [ ]:
# 상관 서브쿼리 — 안쪽이 바깥 행(c.id)을 가리키므로 고객 한 명마다 한 번씩 다시 계산됩니다.
display(run_query("""
SELECT c.name,
       (SELECT count(*) FROM orders o WHERE o.customer_id = c.id)  AS 주문건수,
       (SELECT coalesce(sum(o.amount), 0)
        FROM orders o WHERE o.customer_id = c.id)                  AS 매출
FROM customers c
ORDER BY 매출 DESC
"""))
print("-> 7절의 LEFT JOIN + GROUP BY 와 같은 결과입니다. 같은 답에 이르는 길이 둘인 셈입니다.")

In [ ]:
# 상관 서브쿼리는 UPDATE 에도 그대로 씁니다 — 행마다 다른 값을 채워 넣을 때입니다.
run_sql("""
CREATE TABLE customer_sales (
    customer_id int  PRIMARY KEY,
    name        text NOT NULL,
    total       int  NOT NULL DEFAULT 0
) STRICT
""")
run_sql("INSERT INTO customer_sales (customer_id, name) SELECT id, name FROM customers")

# 지난 노트북에서 본 WHERE 주의의 예외입니다 — 모든 행을 채우는 것이 목적이니까요.
run_sql("""
UPDATE customer_sales
SET total = (SELECT coalesce(sum(o.amount), 0)
             FROM orders o
             WHERE o.customer_id = customer_sales.customer_id)
""")

display(run_query("SELECT * FROM customer_sales ORDER BY total DESC"))

run_sql("DROP TABLE customer_sales")   # 확인이 끝났으니 연습용 표는 치웁니다

### 🖐️ 함께 따라하기 — 평균 평점보다 높은 후기

`reviews` 표에서 **전체 평균 평점보다 높은 평점**을 받은 후기의 `product` · `rating` · `comment` 를 평점 높은 순으로 조회하세요.

**확인 기준**: 3행이 나옵니다.

In [ ]:
# reviews 에서 전체 평균 평점보다 높은 후기를 평점 높은 순으로 조회하세요.

### ✅ 바로 확인 퀴즈

`WHERE amount > (SELECT avg(amount) FROM orders)` 에서 안쪽 질의는 몇 개의 값을 돌려줄까요?

- **A.** 주문 건수만큼
- **B.** 고객 수만큼
- **C.** 정확히 하나
- **D.** 돌려주지 않습니다

<details><summary>정답 보기</summary>

**C** — `avg()` 는 표 전체를 한 값으로 요약합니다. 값이 하나이므로 `>` 같은 비교연산자에 바로 쓸 수 있습니다. 목록이 나오는 서브쿼리에는 `IN` 을 씁니다.

</details>

## 🚀 응용 클론코딩 — 월간 매출 리포트 한 장

지금까지 배운 것을 한 질의에 모읍니다. 아래 조건을 모두 만족하는 리포트를 만드세요.

**요구사항**

1. `customers` 를 왼쪽에 두고 `orders` 를 이어 **주문이 0건인 고객까지** 포함합니다.
2. 고객 이름별로 묶어 `주문건수` · `매출` · `평균주문액` 을 구합니다. 매출이 없는 고객은 매출 `0` 으로 나와야 합니다.
3. **2026년 3월 주문만** 셉니다 (`ordered_at` 이 `'2026-03-01'` 부터 `'2026-03-31'` 사이). 힌트: 7절에서 본 것처럼, 같은 기간 조건도 **어디에 적느냐**에 따라 3월 주문이 없는 고객이 남기도 하고 사라지기도 합니다. 1번 요구사항을 지키려면 어느 쪽일까요?
4. 매출이 큰 순으로 정렬하고, 같으면 이름 오름차순.
5. 마지막에 **전체 합계 한 줄**을 따로 조회해 함께 출력합니다.

**확인 기준**: 고객 5명이 모두 나오고, 3월 매출 합계가 리포트의 매출 합과 같습니다.

In [ ]:
# 위 요구사항대로 월간 매출 리포트와 전체 합계를 만들어 출력하세요.

## 오늘 배운 것

- `SELECT 열 FROM 표 WHERE 조건` — 열을 고르고 행을 거릅니다. `AS` 로 별칭, 열 자리에 계산식도.
- `ORDER BY`(다중 기준·`DESC`) · `LIMIT` · `OFFSET` — 줄 세우고 잘라 가져옵니다.
- `AND`·`OR`·`NOT`(괄호 필수) · `BETWEEN` · `IN` · `LIKE`(`%`·`_`) · `IS NULL` · `DISTINCT`
- `count(*)` 와 `count(열)` 은 다릅니다 — 뒤엣것은 빈 칸을 세지 않습니다.
- `GROUP BY` 로 묶고 `HAVING` 으로 그룹을 거릅니다. `WHERE` 는 묶기 **전**, `HAVING` 은 묶은 **뒤**.
- `SELECT` 는 맨 앞에 적지만 거의 마지막에 실행됩니다 — `WHERE` 에 집계 함수를 못 쓰는 이유이고, 표준 SQL 에서 별칭을 `WHERE` 에 못 쓰는 이유이기도 합니다(별칭은 `ORDER BY` 에서 씁니다).
- `JOIN` 은 짝 있는 행만, `LEFT JOIN` 은 왼쪽을 다 남기고 `NULL` 로 채웁니다 → `coalesce` 로 0.
- `LEFT JOIN` 에서 같은 조건이라도 **`ON` 에 두면 짝을 맺을 때만** 쓰이고, **`WHERE` 에 두면 조인이 끝난 뒤** 걸러서 `NULL` 행까지 사라집니다.
- **인덱스**는 자주 거르는 열에 만들어 두는 찾아보기표입니다 — `CREATE INDEX 이름 ON 표(열)`.
- **서브쿼리**로 질의 결과를 다른 질의의 조건에 끼워 넣습니다. `IN` 의 반대는 `NOT IN` 입니다.
- 집계 결과도 `INSERT INTO 받을표 (열들) SELECT …` 로 표에 그대로 남길 수 있습니다.
- 안쪽 질의가 **바깥 행을 가리키면**(`o.customer_id = c.id`) 행마다 다시 계산됩니다 — **상관 서브쿼리**. `SELECT` 의 열 자리에도, `UPDATE` 의 `SET` 자리에도 씁니다.

## ⏭️ 예고 — 다음 노트북: Supabase 와 pgvector

여기까지는 내 컴퓨터의 파일 하나(sqlite)로 했습니다. 다음 시간엔 **클라우드 PostgreSQL(Supabase)** 에 같은 SQL 을 올려 보고, 그 위에 **pgvector** 를 얹어 "조건으로 좁히고 **의미로** 정렬하는" 검색까지 만듭니다. 지금까지 배운 `WHERE` · `JOIN` 이 그대로 벡터 검색과 한 문장 안에서 만납니다.